In [0]:
import pandas as pd
import re

POSITIVE_WORDS = {
    "great","good","love","loved","loves","excellent","perfect","perfectly","amazing","awesome",
    "happy","best","nice","comfortable","durable","easy","sturdy","soft","reliable",
    "recommend","recommended","fantastic","wonderful","worth","quality","favorite",
    "satisfied","pleased","impressed","gentle","affordable","convenient","cute","adorable",
    "beautiful","superb","outstanding","glad","fast","cozy","comfy","stylish",
    "helpful","efficient","effective","safe","roomy","functional","fits","fun","pretty"
}
NEGATIVE_WORDS = {
    "bad","broke","broken","terrible","awful","poor","disappointed","disappointing",
    "waste","worst","cheap","cheaply","flimsy","useless","defective","returned",
    "return","stopped working","leak","leaks","leaked","smell","odor","stink",
    "uncomfortable","difficult","hard","hate","hated","junk","garbage","regret",
    "annoying","frustrated","frustrating","fail","failed","failure","overpriced",
    "expensive","tiny","small","ripped","tore","tear","noisy","loud","weak",
    "disappointment","horrible","unreliable","unsafe","dangerous"
}
NEGATIONS = {"not", "no", "never", "n't", "without", "barely", "hardly"}

def score_text(text: str):
    """Returns (net_score, hits). net_score = #positive - #negative sentiment
    words found (after negation flipping). Using a raw NET COUNT rather than
    a normalized average, because summaries are short (median 4 words) --
    a normalized average collapses almost every summary to exactly -1, 0,
    or +1, losing the distinction between "good" and "absolutely amazing,
    perfect, love it"."""
    if not isinstance(text, str) or not text.strip():
        return 0, 0
    words = re.findall(r"[a-z']+", text.lower())
    if not words:
        return 0, 0
    net, hits = 0, 0
    for i, w in enumerate(words):
        polarity = 1 if w in POSITIVE_WORDS else (-1 if w in NEGATIVE_WORDS else 0)
        if polarity == 0:
            continue
        window = words[max(0, i-2):i]
        if any(n in window for n in NEGATIONS):
            polarity *= -1
        net += polarity
        hits += 1
    return net, hits

def emoji_for_score(net: int) -> str:
    if net >= 2:
        return "😍"   # strongly positive
    elif net == 1:
        return "😊"   # positive
    elif net == 0:
        return "😐"   # neutral / no sentiment words detected
    elif net == -1:
        return "😞"   # negative
    else:
        return "😡"   # strongly negative (net <= -2)

def run(silver: pd.DataFrame) -> pd.DataFrame:
    df = silver.copy()
    scores = df["summary_clean"].apply(score_text)
    df["summary_sentiment_net"] = scores.apply(lambda t: t[0])
    df["summary_sentiment_hits"] = scores.apply(lambda t: t[1])
    df["summary_emoji"] = df["summary_sentiment_net"].apply(emoji_for_score)

    df.to_pickle("/Workspace/Users/vertubouka@gmail.com/github-databricks/Generative AI project/DbAcademy/silver_reviews_with_emoji.pkl")

    dist = df["summary_emoji"].value_counts(normalize=True).round(3) * 100
    print("[sentiment_emoji] distribution over all reviews:")
    for emo, pct in dist.items():
        print(f"  {emo}  {pct:.1f}%")

    # Agreement check: does text sentiment roughly track the star rating?
    star_bucket = pd.cut(df["overall"], bins=[0,2,3,5], labels=["negative","neutral","positive"])
    emoji_bucket = df["summary_emoji"].map({
        "😍": "positive", "😊": "positive", "😐": "neutral", "😞": "negative", "😡": "negative"
    })
    agreement = (star_bucket.astype(str) == emoji_bucket).mean()
    print(f"[sentiment_emoji] agreement with star rating: {agreement*100:.1f}%")

    disagreements = df[star_bucket.astype(str) != emoji_bucket].sample(3, random_state=42)
    print("\n[sentiment_emoji] example disagreements (rating vs. text sentiment):")
    for _, r in disagreements.iterrows():
        print(f"  {r['overall']}/5 {r['summary_emoji']}  \"{r['summary_clean'][:80]}\"")

    return df

if __name__ == "__main__":
    silver = pd.read_csv("/Workspace/Users/vertubouka@gmail.com/github-databricks/Generative AI project/DbAcademy/silver_table.csv")
    silver["summary_clean"] = silver["summary"].fillna("").str.strip().str.lower()
    result = run(silver)

    # Write the sentiment-enhanced silver table to Unity Catalog
    # so Chunking_doc can read from rag_based_chatbot.default.silver_reviews_with_emoji
    SILVER_EMOJI_TABLE = "rag_based_chatbot.default.silver_reviews_with_emoji"
    # Strip whitespace from column names (bronze table has 'reviewerName ' with trailing space)
    result.columns = result.columns.str.strip()
    spark.createDataFrame(result).write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(SILVER_EMOJI_TABLE)
    print(f"Silver table with emoji saved to UC table: {SILVER_EMOJI_TABLE}")
    print(f"Rows: {result.shape[0]}, Columns: {result.shape[1]}")
